# PTHBV API Exploration

## Overview

This notebook explores the **PTHBV (Precipitation, Temperature for HBV)** gridded dataset from SMHI.
PTHBV provides daily temperature and precipitation on a 4x4 km grid over Sweden since 1961.
The primary purpose is input for hydrological (HBV) modeling.

### PTHBV Specifications

| Attribute | Value |
|-----------|-------|
| **Resolution** | 4 x 4 km |
| **Temporal** | Daily |
| **Coverage** | Swedish land area |
| **Period** | 1961 - present |
| **Projection** | RT90 2.5 gon V (EPSG:3021) |
| **Formats** | NetCDF, JSON, CSV |
| **Variables** | Precipitation (p), Temperature (t) |

### API Entry Point

- Base URL: `https://opendata-download-metanalys.smhi.se`
- Docs (SPA): `https://opendata.smhi.se/pthbv/`
- Old docs: `https://opendata.smhi.se/apidocs/pthbv/`

### Known Categories

| Category | Description |
|----------|-------------|
| `pthbv` | Generic alias |
| `pthbv1gv1` | Version 1 of the grid product |
| `pthbv2gv1` | Version 2 of the grid product |

### API URL Pattern (from SMHI Docusaurus JS bundle)

```
https://opendata-download-metanalys.smhi.se/api/category/{CATEGORY}/version/{VERSION}/geotype/multipoint/from/{YEAR}/to/{YEAR}/period/{PERIOD}/data.json
  ?epsg={EPSG}&ll={LON},{LAT}&var={VAR}
```

---
# Part 1: Setup

In [ ]:
import os
import sys
import json
import requests
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Imports OK")

In [ ]:
# Try importing NetCDF / xarray (optional)
NETCDF_AVAILABLE = False
try:
    import xarray as xr
    import netCDF4
    NETCDF_AVAILABLE = True
    print(f"xarray version: {xr.__version__}")
    print(f"netCDF4 version: {netCDF4.__version__}")
except ImportError as e:
    print(f"NetCDF libraries not available: {e}")
    print("Run: pip install xarray netCDF4")

In [ ]:
# Configuration
BASE_URL = "https://opendata-download-metanalys.smhi.se"

# Known category names discovered from SMHI Docusaurus JS bundle
CATEGORIES = ["pthbv", "pthbv1gv1", "pthbv2gv1"]

# Example point: SMHI headquarters, Norrkoping
EXAMPLE_LON = 16.158
EXAMPLE_LAT = 58.5812

# Output directory
OUTPUT_DIR = Path('../data/raw/pthbv')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base URL: {BASE_URL}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")

---
# Part 2: API Discovery

The PTHBV API lives under `opendata-download-metanalys.smhi.se`. However, the entire
`opendata.smhi.se` domain (and subdomains) now serves a Docusaurus SPA for documentation,
which means many paths return HTML instead of JSON. We systematically probe endpoints to
find which ones return actual data.

In [ ]:
def probe_url(url: str, timeout: int = 30) -> dict:
    """
    Probe a URL and return status code, content type, and a preview of the body.
    """
    try:
        resp = requests.get(url, timeout=timeout)
        content_type = resp.headers.get('Content-Type', 'unknown')
        is_json = 'json' in content_type or (resp.text.strip().startswith('{') or resp.text.strip().startswith('['))
        is_html = 'html' in content_type
        return {
            'url': url,
            'status': resp.status_code,
            'content_type': content_type,
            'is_json': is_json,
            'is_html': is_html,
            'size': len(resp.content),
            'preview': resp.text[:500] if not is_html else '[HTML/SPA page]',
            'response': resp
        }
    except Exception as e:
        return {
            'url': url,
            'status': -1,
            'content_type': 'error',
            'is_json': False,
            'is_html': False,
            'size': 0,
            'preview': str(e),
            'response': None
        }

In [ ]:
# Milestone 1: Probe top-level API endpoints
print("=" * 80)
print("MILESTONE 1: Probe top-level API endpoints")
print("=" * 80)

top_level_urls = [
    f"{BASE_URL}/api.json",
    f"{BASE_URL}/api",
    f"{BASE_URL}/api/version/latest.json",
    f"{BASE_URL}/api/version/1.json",
]

for url in top_level_urls:
    result = probe_url(url)
    status_icon = 'JSON' if result['is_json'] else ('HTML' if result['is_html'] else '???')
    print(f"  [{result['status']}] [{status_icon}] {result['content_type'][:40]}  <- {url}")
    if result['is_json']:
        print(f"         Preview: {result['preview'][:200]}")

In [ ]:
# Milestone 2: Probe category/parameter endpoints for each known category
print("=" * 80)
print("MILESTONE 2: Probe category and parameter endpoints")
print("=" * 80)

for cat in CATEGORIES:
    print(f"\n--- Category: {cat} ---")
    urls = [
        f"{BASE_URL}/api/category/{cat}/version/1/parameter.json",
        f"{BASE_URL}/api/category/{cat}/version/1/parameter",
        f"{BASE_URL}/api/category/{cat}/version/1.json",
    ]
    for url in urls:
        result = probe_url(url)
        status_icon = 'JSON' if result['is_json'] else ('HTML' if result['is_html'] else '???')
        print(f"  [{result['status']}] [{status_icon}] {result['content_type'][:40]}  <- ...{url.split('.smhi.se')[1]}")
        if result['is_json']:
            print(f"         Preview: {result['preview'][:300]}")

In [ ]:
# Milestone 3: Probe data download endpoints (the key ones)
# The web search showed this URL pattern:
# /api/category/{CAT}/version/1/geotype/multipoint/from/{Y}/to/{Y}/period/{P}/data.json
#   ?epsg=4326&ll=LON,LAT&var=p&var=t
#
# Previous probing showed HTTP 406 (Not Acceptable) for .json on data endpoints.
# This means the API exists but may require different format negotiation.

print("=" * 80)
print("MILESTONE 3: Probe data download endpoints")
print("=" * 80)

point_params = f"epsg=4326&ll={EXAMPLE_LON},{EXAMPLE_LAT}&var=p&var=t"

for cat in CATEGORIES:
    print(f"\n--- Category: {cat} ---")
    
    # Try various geotype / period / extension combos
    test_urls = [
        # Point data, JSON
        f"{BASE_URL}/api/category/{cat}/version/1/geotype/multipoint/from/2022/to/2023/period/monthly/data.json?{point_params}",
        # Point data, no extension
        f"{BASE_URL}/api/category/{cat}/version/1/geotype/multipoint/from/2022/to/2023/period/monthly/data?{point_params}",
        # Point data, daily
        f"{BASE_URL}/api/category/{cat}/version/1/geotype/multipoint/from/2022/to/2022/period/daily/data.json?{point_params}",
        # Point data, with Accept header (try separately below)
        # Grid data (NetCDF)
        f"{BASE_URL}/api/category/{cat}/version/1/geotype/grid/from/2022/to/2022/period/monthly/data.netcdf?{point_params}",
    ]
    
    for url in test_urls:
        result = probe_url(url)
        short_path = '...' + url.split('.smhi.se')[1][:100]
        status_icon = 'JSON' if result['is_json'] else ('HTML' if result['is_html'] else result['content_type'][:15])
        print(f"  [{result['status']}] [{status_icon}] {short_path}")
        if result['is_json']:
            print(f"         Preview: {result['preview'][:300]}")
        elif not result['is_html'] and result['status'] == 200:
            print(f"         Content-Type: {result['content_type']}, Size: {result['size']} bytes")

In [ ]:
# Milestone 4: Try with Accept headers instead of file extension
print("=" * 80)
print("MILESTONE 4: Try Accept header negotiation")
print("=" * 80)

cat = "pthbv1gv1"  # primary category
data_url = f"{BASE_URL}/api/category/{cat}/version/1/geotype/multipoint/from/2022/to/2023/period/monthly/data?{point_params}"
param_url = f"{BASE_URL}/api/category/{cat}/version/1/parameter"

accept_types = [
    "application/json",
    "text/csv",
    "application/x-netcdf",
    "*/*",
    "application/xml",
    "text/plain",
]

for accept in accept_types:
    print(f"\n  Accept: {accept}")
    for label, url in [("parameter", param_url), ("data", data_url)]:
        try:
            resp = requests.get(url, headers={"Accept": accept}, timeout=30)
            ct = resp.headers.get('Content-Type', 'unknown')
            is_json = resp.text.strip()[:1] in ('{', '[')
            preview = resp.text[:150] if is_json else ('[HTML]' if 'html' in ct else resp.text[:150])
            print(f"    [{resp.status_code}] {label:12s} -> {ct[:50]}  {preview[:100]}")
        except Exception as e:
            print(f"    [ERR] {label:12s} -> {e}")

In [ ]:
# Milestone 5: Explore additional URL path variations
# The SMHI docs JS showed these doc pages: get_point, get_all_points
# Try geotype variations: point, multipoint, grid

print("=" * 80)
print("MILESTONE 5: Explore geotype and path variations")
print("=" * 80)

cat = "pthbv1gv1"
geotypes = ["point", "multipoint", "grid"]
periods = ["daily", "monthly", "latest-day", "latest-months"]
extensions = [".json", ".csv", ".nc", ".netcdf", ""]

results = []
for geo in geotypes:
    for period in periods:
        for ext in extensions:
            url = f"{BASE_URL}/api/category/{cat}/version/1/geotype/{geo}/from/2022/to/2022/period/{period}/data{ext}?{point_params}"
            try:
                resp = requests.get(url, timeout=15)
                ct = resp.headers.get('Content-Type', 'unknown')
                is_html = 'html' in ct
                results.append({
                    'geotype': geo,
                    'period': period,
                    'ext': ext or '(none)',
                    'status': resp.status_code,
                    'content_type': ct.split(';')[0],
                    'is_html': is_html,
                    'size': len(resp.content)
                })
            except Exception as e:
                results.append({
                    'geotype': geo, 'period': period, 'ext': ext or '(none)',
                    'status': -1, 'content_type': str(e)[:40], 'is_html': False, 'size': 0
                })
            time.sleep(0.2)  # be polite

df_results = pd.DataFrame(results)
print(f"\nTotal probes: {len(df_results)}")
print(f"\nStatus code distribution:")
print(df_results['status'].value_counts().to_string())
print(f"\nContent type distribution:")
print(df_results['content_type'].value_counts().to_string())

# Show non-HTML, non-error results (likely real API responses)
interesting = df_results[(~df_results['is_html']) & (df_results['status'] > 0)]
if len(interesting) > 0:
    print(f"\nInteresting (non-HTML) responses:")
    print(interesting.to_string(index=False))
else:
    print("\nAll responses returned HTML (Docusaurus SPA) - API may require different approach.")

In [ ]:
# Show full results table
print("\nFull results table:")
print(df_results.to_string(index=False))

---
# Part 3: Try Known Working Patterns

If the standard REST endpoints all return the Docusaurus SPA, the PTHBV data may only be
available through:
1. A different subdomain or path
2. The SMHI grid archive (like MESAN uses `opendata-download-grid-archive.smhi.se`)
3. Direct file download links from the SMHI data page

In [ ]:
# Milestone 6: Try alternative base URLs and subdomains
print("=" * 80)
print("MILESTONE 6: Alternative base URLs")
print("=" * 80)

alternative_bases = [
    "https://opendata-download-metanalys.smhi.se",
    "https://opendata-download-grid-archive.smhi.se",
    "https://opendata-download-grid.smhi.se",
    "https://opendata.smhi.se",
]

for base in alternative_bases:
    print(f"\n--- {base} ---")
    test_paths = [
        "/api.json",
        "/api/version/latest.json",
        "/api/category/pthbv1gv1/version/1/parameter.json",
    ]
    for path in test_paths:
        result = probe_url(f"{base}{path}")
        kind = 'JSON' if result['is_json'] else ('HTML' if result['is_html'] else result['content_type'][:20])
        print(f"  [{result['status']}] [{kind}]  {path}")
        if result['is_json']:
            print(f"       {result['preview'][:200]}")

In [ ]:
# Milestone 7: Try the metobs-style API (known working) to compare patterns
# The metobs API at opendata-download-metobs.smhi.se definitely works
print("=" * 80)
print("MILESTONE 7: Reference - metobs API (known working)")
print("=" * 80)

metobs_url = "https://opendata-download-metobs.smhi.se/api/version/latest.json"
result = probe_url(metobs_url)
print(f"  [{result['status']}] Content-Type: {result['content_type']}")
if result['is_json']:
    data = result['response'].json()
    print(f"  Keys: {list(data.keys())}")
    if 'resource' in data:
        print(f"  Number of parameters: {len(data['resource'])}")
        # Show first few parameters
        for r in data['resource'][:5]:
            print(f"    key={r['key']}: {r['title']} ({r.get('unit', 'N/A')})")
else:
    print(f"  Not JSON: {result['preview'][:200]}")

In [ ]:
# Milestone 8: Attempt to download a sample NetCDF file
# Based on SMHI docs, the full grid download returns NetCDF4
print("=" * 80)
print("MILESTONE 8: Attempt NetCDF grid download")
print("=" * 80)

cat = "pthbv1gv1"
netcdf_urls = [
    f"{BASE_URL}/api/category/{cat}/version/1/geotype/grid/from/2022/to/2022/period/monthly/data.netcdf",
    f"{BASE_URL}/api/category/{cat}/version/1/geotype/grid/from/2022/to/2022/period/daily/data.netcdf",
    f"{BASE_URL}/api/category/{cat}/version/1/geotype/grid/from/2022/to/2022/data.netcdf",
]

for url in netcdf_urls:
    result = probe_url(url)
    ct = result['content_type']
    short = url.split('.smhi.se')[1]
    print(f"  [{result['status']}] {ct[:50]}  Size: {result['size']}B  <- {short}")
    
    # If we got a non-HTML response, save it
    if not result['is_html'] and result['status'] == 200 and result['size'] > 2000:
        outfile = OUTPUT_DIR / f"pthbv_sample.nc"
        with open(outfile, 'wb') as f:
            f.write(result['response'].content)
        print(f"         Saved to {outfile} ({result['size']} bytes)")
        break

---
# Part 4: Parse and Explore Data (if available)

In [ ]:
# Milestone 9: If we got any JSON data, parse and display it
print("=" * 80)
print("MILESTONE 9: Parse any successful JSON responses")
print("=" * 80)

# Try the point data endpoint with various categories
for cat in CATEGORIES:
    url = f"{BASE_URL}/api/category/{cat}/version/1/geotype/multipoint/from/2022/to/2022/period/monthly/data.json?epsg=4326&ll={EXAMPLE_LON},{EXAMPLE_LAT}&var=p&var=t"
    result = probe_url(url)
    
    if result['is_json'] and result['response'] is not None:
        print(f"\nGot JSON from category '{cat}':")
        data = result['response'].json()
        print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])
        
        # Try to convert to DataFrame
        if isinstance(data, list):
            df = pd.DataFrame(data)
            print(f"\nDataFrame shape: {df.shape}")
            print(df.head())
        elif isinstance(data, dict) and 'timeSeries' in data:
            print(f"\nTime series entries: {len(data['timeSeries'])}")
            print("First entry:", json.dumps(data['timeSeries'][0], indent=2)[:500])
    else:
        print(f"  [{result['status']}] Category '{cat}': No JSON response")

In [ ]:
# Milestone 10: If we got a NetCDF file, explore it
print("=" * 80)
print("MILESTONE 10: Explore NetCDF data (if downloaded)")
print("=" * 80)

nc_file = OUTPUT_DIR / "pthbv_sample.nc"

if nc_file.exists() and NETCDF_AVAILABLE:
    print(f"Opening {nc_file} ({nc_file.stat().st_size} bytes)")
    try:
        ds = xr.open_dataset(nc_file)
        print(f"\nDataset info:")
        print(ds)
        print(f"\nVariables: {list(ds.data_vars)}")
        print(f"Coordinates: {list(ds.coords)}")
        print(f"Dimensions: {dict(ds.dims)}")
        print(f"\nAttributes:")
        for k, v in ds.attrs.items():
            print(f"  {k}: {v}")
    except Exception as e:
        print(f"Error reading NetCDF: {e}")
elif nc_file.exists():
    print(f"NetCDF file exists ({nc_file.stat().st_size} bytes) but xarray/netCDF4 not available")
else:
    print("No NetCDF file downloaded yet.")

---
# Part 5: Summary of Findings

In [ ]:
print("=" * 80)
print("SUMMARY OF PTHBV API EXPLORATION")
print("=" * 80)

print("""
Known facts:
  - API base: https://opendata-download-metanalys.smhi.se
  - Categories: pthbv, pthbv1gv1, pthbv2gv1
  - Projection: RT90 2.5 gon V (EPSG:3021)
  - Grid: 4x4 km over Swedish land
  - Variables: p (precipitation), t (temperature)
  - Period: 1961 - present, daily
  - URL template:
    /api/category/{CAT}/version/1/geotype/{GEO}/from/{YEAR}/to/{YEAR}/period/{PERIOD}/data.{FMT}
    ?epsg=4326&ll=LON,LAT&var=p&var=t

Issue encountered:
  - The opendata-download-metanalys.smhi.se domain currently serves a Docusaurus SPA
    for ALL paths, returning text/html instead of JSON/NetCDF.
  - HTTP 406 on data endpoints suggests the API exists behind the SPA but format
    negotiation may have changed.
  - The SPA is a client-side rendered documentation portal that intercepts all routes.

Next steps:
  1. Check if SMHI has migrated the API to a new endpoint
  2. Try the SMHI Python client library (pip install smhi-open-data)
  3. Contact SMHI support for current API access
  4. Check the grid archive approach (like MESAN uses)
  5. Use the SMHI web interface to manually download bulk data
""")

---
# Part 6: Alternative - Use SMHI Python Client

If the direct API is blocked by the SPA, try the `smhi-open-data` Python package
which may handle the API negotiation differently.

In [ ]:
# Try the smhi-open-data package
try:
    from smhi.smhi_lib import Smhi
    print("smhi-open-data package available")
    # This package typically handles the metobs API
    # It may not support PTHBV directly
except ImportError:
    print("smhi-open-data not installed. Run: pip install smhi-open-data")
    print("Note: this package may only support the metobs API, not PTHBV gridded data.")

In [ ]:
# Alternative: Try downloading PTHBV data from the SMHI data download page
# The page https://www.smhi.se/data/meteorologi/nederbord-och-temperatur/griddade-nederbords-och-temperaturdata
# may have direct download links for bulk data files

print("Alternative download approach:")
print("  Visit: https://www.smhi.se/data/meteorologi/nederbord-och-temperatur/griddade-nederbords-och-temperaturdata")
print("  This page may offer direct file downloads for PTHBV gridded data.")
print("  Look for NetCDF or CSV bulk download options.")